# CLARION — ScanShield Training Notebook
## EfficientNet-B0 Currency Note Authenticity Classifier

**CLARION Threat Intelligence Platform**

This notebook trains the ScanShield computer vision model to classify Indian currency notes as GENUINE or FAKE.

### What this notebook does:
1. Installs required libraries
2. Lets you upload genuine ₹500 / ₹2000 note images
3. Generates 500+ synthetic fake variants using image augmentation
4. Trains EfficientNet-B0 in two phases (frozen base → fine-tuning)
5. Evaluates and prints precision/recall/F1/AUC-ROC
6. Saves and downloads the model file

### Your only job:
- Upload 15-20 clear photos of genuine ₹500 and/or ₹2000 notes in Cell 3
- Download high-quality note images from: https://www.rbi.org.in/scripts/ic_currencynotes.aspx
- Run all cells top to bottom (Runtime → Run all)

> ⏱️ Estimated time: ~20-30 minutes on Colab GPU (T4)

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────
print('Installing dependencies...')
!pip install -q tensorflow==2.16.1 opencv-python-headless pillow numpy scikit-learn tqdm matplotlib
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os, io, random, shutil, math
import numpy as np
import cv2
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model, callbacks
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

# Set seeds for reproducibility
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__} | GPUs available: {len(gpus)}')

# Directories
BASE_DIR   = Path('/content/clarion_currency')
GENUINE_DIR = BASE_DIR / 'genuine'
FAKE_DIR    = BASE_DIR / 'fake'
TRAIN_DIR   = BASE_DIR / 'dataset/train'
VAL_DIR     = BASE_DIR / 'dataset/val'
TEST_DIR    = BASE_DIR / 'dataset/test'

for d in [GENUINE_DIR, FAKE_DIR, TRAIN_DIR/'genuine', TRAIN_DIR/'fake',
          VAL_DIR/'genuine', VAL_DIR/'fake', TEST_DIR/'genuine', TEST_DIR/'fake']:
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
print('✅ Directories created')

In [ ]:
# ── Cell 3: Upload Genuine Note Images ────────────────────────────────────────
# Upload 15-20 clear photos of genuine Rs 500 or Rs 2000 notes.
# Photos can be taken with a phone camera — good lighting, flat surface.
# You can also download RBI official images from:
# https://www.rbi.org.in/scripts/ic_currencynotes.aspx

from google.colab import files
print('Please select your genuine note images (JPEG/PNG)...')
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = GENUINE_DIR / filename
    dest.write_bytes(content)
    print(f'  Saved: {filename}')

genuine_count = len(list(GENUINE_DIR.glob('*.jpg'))) + len(list(GENUINE_DIR.glob('*.png')))
print(f'\n✅ {genuine_count} genuine images ready for augmentation')

In [ ]:
# ── Cell 4: Generate Synthetic Fake Images ─────────────────────────────────────
# Creates realistic fake variants by:
# a) Blurring the security thread region (x: 38-52% of width)
# b) Shifting hue by +/-20 degrees (simulates poor printing)
# c) Elastic distortion on watermark zone (left 30%)
# d) Reducing brightness/sharpness of serial number region

def blur_security_thread(img):
    """Blur the vertical security thread (38-52% of image width)."""
    h, w = img.shape[:2]
    x1, x2 = int(w * 0.38), int(w * 0.52)
    result = img.copy()
    blur_strength = random.choice([15, 21, 31])
    result[:, x1:x2] = cv2.GaussianBlur(img[:, x1:x2], (blur_strength, blur_strength), 0)
    return result

def shift_hue(img, shift_deg=None):
    """Shift hue by random degrees to simulate ink colour mismatch."""
    if shift_deg is None:
        shift_deg = random.randint(-25, 25)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int32)
    hsv[:,:,0] = (hsv[:,:,0] + shift_deg) % 180
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def distort_watermark(img):
    """Apply elastic distortion to watermark region (left 30%)."""
    h, w = img.shape[:2]
    x2 = int(w * 0.30)
    region = img[:, :x2].copy()
    # Random displacement field
    dx = cv2.GaussianBlur((np.random.rand(*region.shape[:2]) * 2 - 1).astype(np.float32), (17, 17), 5) * 8
    dy = cv2.GaussianBlur((np.random.rand(*region.shape[:2]) * 2 - 1).astype(np.float32), (17, 17), 5) * 8
    grid_x, grid_y = np.meshgrid(np.arange(region.shape[1]), np.arange(region.shape[0]))
    map_x = (grid_x + dx).astype(np.float32)
    map_y = (grid_y + dy).astype(np.float32)
    distorted = cv2.remap(region, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    result = img.copy()
    result[:, :x2] = distorted
    return result

def degrade_serial_number(img):
    """Blur and darken the serial number zone."""
    h, w = img.shape[:2]
    x1, x2 = int(w * 0.55), int(w * 0.95)
    y1, y2 = int(h * 0.70), int(h * 0.90)
    result = img.copy()
    result[y1:y2, x1:x2] = cv2.GaussianBlur(img[y1:y2, x1:x2], (7, 7), 0)
    result[y1:y2, x1:x2] = (result[y1:y2, x1:x2] * random.uniform(0.7, 0.9)).astype(np.uint8)
    return result

def augment_image(img):
    """Apply random augmentation (rotation, brightness, noise)."""
    h, w = img.shape[:2]
    # Random rotation
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    # Brightness
    factor = random.uniform(0.8, 1.2)
    img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    # Gaussian noise
    noise_sigma = random.uniform(0, 10)
    if noise_sigma > 2:
        noise = np.random.normal(0, noise_sigma, img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return img

# Generate fake images
genuine_images = list(GENUINE_DIR.glob('*.jpg')) + list(GENUINE_DIR.glob('*.png'))
FAKES_PER_GENUINE = max(3, 500 // len(genuine_images))  # target ~500 fakes total

print(f'Generating synthetic fakes ({FAKES_PER_GENUINE} per genuine image)...')
fake_count = 0
for genuine_path in tqdm(genuine_images):
    img = cv2.imread(str(genuine_path))
    if img is None:
        continue
    img = cv2.resize(img, (640, 320))

    for i in range(FAKES_PER_GENUINE):
        fake = img.copy()
        # Apply 2-3 random forgery techniques
        techniques = random.sample([blur_security_thread, shift_hue, distort_watermark, degrade_serial_number], k=random.randint(2, 3))
        for technique in techniques:
            fake = technique(fake)
        fake = augment_image(fake)
        out_path = FAKE_DIR / f'fake_{genuine_path.stem}_{i:03d}.jpg'
        cv2.imwrite(str(out_path), fake, [cv2.IMWRITE_JPEG_QUALITY, 90])
        fake_count += 1

# Also augment genuine images
aug_count = 0
for genuine_path in tqdm(genuine_images, desc='Augmenting genuine images'):
    img = cv2.imread(str(genuine_path))
    if img is None:
        continue
    img = cv2.resize(img, (640, 320))
    for i in range(FAKES_PER_GENUINE):
        aug = augment_image(img.copy())
        out_path = GENUINE_DIR / f'aug_{genuine_path.stem}_{i:03d}.jpg'
        cv2.imwrite(str(out_path), aug, [cv2.IMWRITE_JPEG_QUALITY, 90])
        aug_count += 1

total_genuine = len(list(GENUINE_DIR.glob('*.jpg')))
total_fake    = len(list(FAKE_DIR.glob('*.jpg')))
print(f'\n✅ Dataset generated:')
print(f'   Genuine: {total_genuine} images')
print(f'   Fake:    {total_fake} images')
print(f'   Total:   {total_genuine + total_fake} images')

In [ ]:
# ── Cell 5: Build Train/Val/Test Splits ───────────────────────────────────────
def split_images(src_dir, train_d, val_d, test_d, class_name, splits=(0.8, 0.1, 0.1)):
    images = list(Path(src_dir).glob('*.jpg'))
    random.shuffle(images)
    n = len(images)
    n_train = int(n * splits[0])
    n_val   = int(n * splits[1])
    for i, img_path in enumerate(images):
        if i < n_train:       dest = Path(train_d) / class_name
        elif i < n_train+n_val: dest = Path(val_d) / class_name
        else:                   dest = Path(test_d) / class_name
        shutil.copy(img_path, dest / img_path.name)
    return n_train, n_val, n - n_train - n_val

g_train, g_val, g_test = split_images(GENUINE_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR, 'genuine')
f_train, f_val, f_test = split_images(FAKE_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR, 'fake')

print(f'Split complete:')
print(f'  Train: {g_train} genuine + {f_train} fake = {g_train+f_train} total')
print(f'  Val:   {g_val} genuine + {f_val} fake = {g_val+f_val} total')
print(f'  Test:  {g_test} genuine + {f_test} fake = {g_test+f_test} total')

In [ ]:
# ── Cell 6: Build Model ────────────────────────────────────────────────────────
BATCH_SIZE = 16

# Data generators
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    brightness_range=[0.85, 1.15],
    horizontal_flip=False,  # Currency notes are orientation-specific
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary')

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=1, class_mode='binary', shuffle=False)

# Build EfficientNetB0 model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # Phase 1: freeze base

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

print(f'Model built. Parameters: {model.count_params():,}')
print(f'  Trainable:     {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')
print(f'  Non-trainable: {sum(tf.size(v).numpy() for v in model.non_trainable_variables):,}')

In [ ]:
# ── Cell 7: Phase 1 Training (Frozen Base) ─────────────────────────────────────
print('Phase 1: Training classification head (base frozen)...')
cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=cb_list,
    verbose=1,
)
print(f'Phase 1 best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# ── Cell 8: Phase 2 Fine-tuning (Unfreeze top 20 layers) ─────────────────────
print('Phase 2: Fine-tuning top 20 layers of EfficientNet...')
base_model.trainable = True
# Freeze all except last 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=cb_list,
    verbose=1,
)
print(f'Phase 2 best val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

In [ ]:
# ── Cell 9: Evaluation ────────────────────────────────────────────────────────
print('Evaluating on test set...')
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=1).flatten()
y_true = test_gen.classes
y_pred = (y_pred_probs > 0.5).astype(int)

print('\n' + '='*50)
print('CLARION ScanShield — Evaluation Results')
print('='*50)
print(classification_report(y_true, y_pred, target_names=['Genuine', 'Fake']))
print(f'AUC-ROC: {roc_auc_score(y_true, y_pred_probs):.4f}')

cm = confusion_matrix(y_true, y_pred)
print(f'Confusion Matrix:')
print(f'  True Genuine (correctly identified): {cm[0][0]}')
print(f'  False Fake   (genuine wrongly flagged): {cm[0][1]}')
print(f'  False Genuine (fake missed): {cm[1][0]}')
print(f'  True Fake    (correctly caught): {cm[1][1]}')

In [ ]:
# ── Cell 10: Save and Download Model ──────────────────────────────────────────
SAVE_PATH = '/content/scan_efficientnet.h5'
model.save(SAVE_PATH)
print(f'Model saved to {SAVE_PATH}')

import os
size_mb = os.path.getsize(SAVE_PATH) / (1024*1024)
print(f'Model size: {size_mb:.1f} MB')

print('\nDownloading model file...')
from google.colab import files
files.download(SAVE_PATH)

print('\n' + '='*60)
print('DONE! Next steps:')
print('1. Move the downloaded scan_efficientnet.h5 to:')
print('   d:/Projects/ET-AI/CLARION/backend/saved_models/')
print('2. Restart the backend (uvicorn main:app --reload)')
print('3. The backend will auto-detect and load the real model')
print('='*60)